# W2-D1: Alert Correlation — Từ Noise Sang Signal

**Mục tiêu:** Xây dựng pipeline 4-layer để gom 20 alert thành các cluster có ý nghĩa.

| Layer | Chức năng |
|---|---|
| Layer 1: Dedup | Gộp alert trùng lặp bằng fingerprint |
| Layer 2: Time-Window | Chia alert thành session (gap_sec) |
| Layer 3: Topology | Gom theo khoảng cách service trên graph |
| Layer 4: Semantic | (Bonus) Gom alert cùng chủ đề |

**Parameters:** `gap_sec=120`, `max_hop=2`

## 1. Load Dataset

In [1]:
import json
import os
from datetime import datetime, timezone
from collections import defaultdict

# --- Load alerts ---
with open('dataset/alerts_sample.jsonl', 'r', encoding='utf-8') as f:
    raw_alerts = [json.loads(line) for line in f if line.strip()]

# --- Load service graph ---
with open('dataset/services.json', 'r', encoding='utf-8') as f:
    services_data = json.load(f)

print(f"Loaded {len(raw_alerts)} alerts")
print(f"Loaded {len(services_data['services'])} services, {len(services_data['stores'])} stores, {len(services_data['edges'])} edges")
print()
print("=== Sample Alert ===")
print(json.dumps(raw_alerts[0], indent=2))

Loaded 20 alerts
Loaded 10 services, 4 stores, 17 edges

=== Sample Alert ===
{
  "id": "a-0001",
  "ts": "2026-06-12T09:42:01Z",
  "service": "payment-svc",
  "metric": "db_connection_pool_used_ratio",
  "severity": "warn",
  "value": 0.85,
  "threshold": 0.8,
  "labels": {
    "env": "prod",
    "region": "ap-southeast-1"
  }
}


## 2. Khám phá dữ liệu Alert

In [2]:
# Phân tích phân bố service và severity
from collections import Counter

service_counts = Counter(a['service'] for a in raw_alerts)
severity_counts = Counter(a['severity'] for a in raw_alerts)
metric_counts = Counter(a['metric'] for a in raw_alerts)

print("=== Phân bố theo Service ===")
for svc, count in service_counts.most_common():
    print(f"  {svc:25s} : {count} alerts")

print(f"\n=== Phân bố theo Severity ===")
for sev, count in severity_counts.most_common():
    print(f"  {sev:10s} : {count} alerts")

print(f"\n=== Phân bố theo Metric ===")
for met, count in metric_counts.most_common():
    print(f"  {met:40s} : {count} alerts")

# Timeline
print(f"\n=== Timeline ===")
print(f"  First alert: {raw_alerts[0]['ts']}")
print(f"  Last alert:  {raw_alerts[-1]['ts']}")

# Tính gaps giữa các alert liên tiếp
def parse_ts(ts_str):
    return datetime.fromisoformat(ts_str.replace('Z', '+00:00'))

print(f"\n=== Gaps giữa các alert liên tiếp (giây) ===")
sorted_alerts = sorted(raw_alerts, key=lambda a: a['ts'])
gaps = []
for i in range(1, len(sorted_alerts)):
    gap = (parse_ts(sorted_alerts[i]['ts']) - parse_ts(sorted_alerts[i-1]['ts'])).total_seconds()
    gaps.append(gap)
    print(f"  {sorted_alerts[i-1]['id']} -> {sorted_alerts[i]['id']}: {gap:.0f}s")

print(f"\n  Max gap: {max(gaps):.0f}s | Mean gap: {sum(gaps)/len(gaps):.1f}s | Median: {sorted(gaps)[len(gaps)//2]:.0f}s")

=== Phân bố theo Service ===
  payment-svc               : 8 alerts
  checkout-svc              : 4 alerts
  edge-lb                   : 3 alerts
  notification-svc          : 2 alerts
  cart-svc                  : 1 alerts
  recommender-svc           : 1 alerts
  search-svc                : 1 alerts

=== Phân bố theo Severity ===
  crit       : 12 alerts
  warn       : 8 alerts

=== Phân bố theo Metric ===
  latency_p99_ms                           : 6 alerts
  db_connection_pool_used_ratio            : 3 alerts
  error_rate                               : 2 alerts
  upstream_5xx_rate                        : 2 alerts
  downstream_payment_error_rate            : 1 alerts
  queue_lag_ms                             : 1 alerts
  request_drop_rate                        : 1 alerts
  cpu_utilization                          : 1 alerts
  p99_latency_ms                           : 1 alerts
  catalog_db_query_time_ms                 : 1 alerts
  queue_depth                              : 1 al

## 3. Layer 1 — Dedup (Khử trùng lặp)

In [3]:
from correlate import fingerprint, Deduper

# Chạy Dedup
deduper = Deduper()
for alert in raw_alerts:
    deduper.push(alert)

print(f"=== Layer 1: Dedup ===")
print(f"  Input:  {len(raw_alerts)} alerts")
print(f"  Unique fingerprints: {len(deduper.store)}")
print(f"  Duplicates removed: {len(raw_alerts) - len(deduper.store)}")
print()

# Hiển thị các fingerprint bị trùng
print("=== Fingerprints có trùng lặp ===")
for fp, info in deduper.store.items():
    if info['count'] > 1:
        alert_ids = [a['id'] for a in info['alerts']]
        print(f"  {fp}")
        print(f"    → Lặp {info['count']} lần: {alert_ids}")
        print(f"    → first_seen: {info['first_seen']}, last_seen: {info['last_seen']}")

print()
print("=== Tất cả Fingerprints ===")
for fp, info in sorted(deduper.store.items()):
    print(f"  [{info['count']}x] {fp}")

=== Layer 1: Dedup ===
  Input:  20 alerts
  Unique fingerprints: 17
  Duplicates removed: 3

=== Fingerprints có trùng lặp ===
  payment-svc|db_connection_pool_used_ratio|crit
    → Lặp 2 lần: ['a-0002', 'a-0011']
    → first_seen: 2026-06-12T09:42:18Z, last_seen: 2026-06-12T09:44:02Z
  payment-svc|latency_p99_ms|crit
    → Lặp 3 lần: ['a-0003', 'a-0008', 'a-0015']
    → first_seen: 2026-06-12T09:42:22Z, last_seen: 2026-06-12T09:46:01Z

=== Tất cả Fingerprints ===
  [1x] cart-svc|latency_p99_ms|warn
  [1x] checkout-svc|downstream_payment_error_rate|crit
  [1x] checkout-svc|latency_p99_ms|crit
  [1x] checkout-svc|latency_p99_ms|warn
  [1x] checkout-svc|request_drop_rate|crit
  [1x] edge-lb|p99_latency_ms|crit
  [1x] edge-lb|upstream_5xx_rate|crit
  [1x] edge-lb|upstream_5xx_rate|warn
  [1x] notification-svc|queue_depth|crit
  [1x] notification-svc|queue_lag_ms|warn
  [2x] payment-svc|db_connection_pool_used_ratio|crit
  [1x] payment-svc|db_connection_pool_used_ratio|warn
  [1x] payment

## 4. Layer 2 — Time-Window (Session Window)

In [4]:
from correlate import session_groups

GAP_SEC = 120  # Sweet spot cho production (2 phút)

# Lấy danh sách alert đã dedup (giữ tất cả, đánh dấu fingerprint)
deduped_alerts = deduper.get_deduped_alerts()
sessions = session_groups(deduped_alerts, gap_sec=GAP_SEC)

print(f"=== Layer 2: Session Window (gap_sec={GAP_SEC}) ===")
print(f"  Input:  {len(deduped_alerts)} alerts")
print(f"  Output: {len(sessions)} session(s)")
print()

for i, session in enumerate(sessions):
    services = sorted(set(a['service'] for a in session))
    ts_range = [min(a['ts'] for a in session), max(a['ts'] for a in session)]
    print(f"  Session {i}: {len(session)} alerts")
    print(f"    Time range: {ts_range[0]} → {ts_range[1]}")
    print(f"    Services: {services}")
    print()

# So sánh gap_sec=30 vs gap_sec=120 vs gap_sec=600
print("=== So sánh gap_sec ===")
for g in [30, 60, 120, 300, 600]:
    sess = session_groups(deduped_alerts, gap_sec=g)
    print(f"  gap_sec={g:4d} → {len(sess)} session(s)")

=== Layer 2: Session Window (gap_sec=120) ===
  Input:  20 alerts
  Output: 1 session(s)

  Session 0: 20 alerts
    Time range: 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
    Services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']

=== So sánh gap_sec ===
  gap_sec=  30 → 5 session(s)
  gap_sec=  60 → 1 session(s)
  gap_sec= 120 → 1 session(s)
  gap_sec= 300 → 1 session(s)
  gap_sec= 600 → 1 session(s)


## 5. Layer 3 — Topology (Service Graph)

In [5]:
import networkx as nx
from correlate import build_service_graph, topology_group

graph = build_service_graph(services_data)
undirected = graph.to_undirected()

print(f"=== Service Graph ===")
print(f"  Nodes: {graph.number_of_nodes()}")
print(f"  Edges: {graph.number_of_edges()}")
print()

# Hiển thị khoảng cách giữa các service có alert
alert_services = sorted(set(a['service'] for a in raw_alerts))
print(f"=== Khoảng cách (hops) giữa các service có alert ===")

# Header
header = f"{'':20s}"
for s in alert_services:
    header += f" {s[:8]:>8s}"
print(header)

# Distance matrix
for s1 in alert_services:
    row = f"{s1:20s}"
    for s2 in alert_services:
        if s1 == s2:
            row += f" {'·':>8s}"
        else:
            try:
                d = nx.shortest_path_length(undirected, s1, s2)
                marker = ' ✓' if d <= 2 else ''
                row += f" {str(d)+marker:>8s}"
            except nx.NetworkXNoPath:
                row += f" {'∞':>8s}"
    print(row)

print(f"\n  (✓ = within max_hop=2)")

# Chạy topology grouping trên toàn bộ alert
MAX_HOP = 2
topo_groups = topology_group(raw_alerts, graph, max_hop=MAX_HOP)
print(f"\n=== Topology Grouping (max_hop={MAX_HOP}) ===")
print(f"  Input: {len(raw_alerts)} alerts from {len(alert_services)} services")
print(f"  Output: {len(topo_groups)} group(s)")
for i, group in enumerate(topo_groups):
    svcs = sorted(set(a['service'] for a in group))
    print(f"  Group {i}: {len(group)} alerts | services={svcs}")

=== Service Graph ===
  Nodes: 14
  Edges: 17

=== Khoảng cách (hops) giữa các service có alert ===
                     cart-svc checkout  edge-lb notifica payment- recommen search-s
cart-svc                    ·      1 ✓      2 ✓      2 ✓      2 ✓      2 ✓        3
checkout-svc              1 ✓        ·      1 ✓      1 ✓      1 ✓        3      2 ✓
edge-lb                   2 ✓      1 ✓        ·      2 ✓      2 ✓      2 ✓      1 ✓
notification-svc          2 ✓      1 ✓      2 ✓        ·      2 ✓        4        3
payment-svc               2 ✓      1 ✓      2 ✓      2 ✓        ·        4        3
recommender-svc           2 ✓        3      2 ✓        4        4        ·      2 ✓
search-svc                  3      2 ✓      1 ✓        3        3      2 ✓        ·

  (✓ = within max_hop=2)

=== Topology Grouping (max_hop=2) ===
  Input: 20 alerts from 7 services
  Output: 3 group(s)
  Group 0: 18 alerts | services=['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']


## 6. Layer 4 (Bonus) — Semantic Similarity

In [6]:
from correlate import text_similarity, semantic_merge

# Hiển thị similarity matrix cho các alert cùng service
print("=== Semantic Similarity (cùng service) ===")
print("  Chỉ tính giữa alert CÙNG service để tránh gom nhầm cross-service.")
print()

# Nhóm alert theo service
by_service = defaultdict(list)
for a in raw_alerts:
    by_service[a['service']].append(a)

for svc, alerts in sorted(by_service.items()):
    if len(alerts) < 2:
        continue
    print(f"  [{svc}]")
    for i in range(len(alerts)):
        for j in range(i+1, len(alerts)):
            sim = text_similarity(alerts[i], alerts[j])
            if sim > 0:
                marker = ' ← MERGE' if sim >= 0.4 else ''
                print(f"    {alerts[i]['id']} ({alerts[i]['metric']})")
                print(f"    {alerts[j]['id']} ({alerts[j]['metric']})")
                print(f"    → Jaccard similarity = {sim:.3f}{marker}")
                print()

=== Semantic Similarity (cùng service) ===
  Chỉ tính giữa alert CÙNG service để tránh gom nhầm cross-service.

  [checkout-svc]
    a-0005 (latency_p99_ms)
    a-0017 (latency_p99_ms)
    → Jaccard similarity = 1.000 ← MERGE

    a-0006 (downstream_payment_error_rate)
    a-0012 (request_drop_rate)
    → Jaccard similarity = 0.167

  [edge-lb]
    a-0007 (upstream_5xx_rate)
    a-0020 (upstream_5xx_rate)
    → Jaccard similarity = 1.000 ← MERGE

  [notification-svc]
    a-0010 (queue_lag_ms)
    a-0019 (queue_depth)
    → Jaccard similarity = 0.250

  [payment-svc]
    a-0001 (db_connection_pool_used_ratio)
    a-0002 (db_connection_pool_used_ratio)
    → Jaccard similarity = 1.000 ← MERGE

    a-0001 (db_connection_pool_used_ratio)
    a-0011 (db_connection_pool_used_ratio)
    → Jaccard similarity = 1.000 ← MERGE

    a-0002 (db_connection_pool_used_ratio)
    a-0011 (db_connection_pool_used_ratio)
    → Jaccard similarity = 1.000 ← MERGE

    a-0003 (latency_p99_ms)
    a-0008 (lat

## 7. Chạy Full Pipeline — correlate()

In [7]:
from correlate import correlate, build_summary

GAP_SEC = 120
MAX_HOP = 2

# Chạy pipeline
clusters = correlate(raw_alerts, graph, gap_sec=GAP_SEC, max_hop=MAX_HOP)
summary = build_summary(raw_alerts, clusters)

print(f"{'='*60}")
print(f" CORRELATION RESULTS (gap_sec={GAP_SEC}, max_hop={MAX_HOP})")
print(f"{'='*60}")
print(f"  Input alerts:    {summary['input_alerts']}")
print(f"  Output clusters: {summary['output_clusters']}")
print(f"  Reduction ratio: {summary['reduction_ratio']} ({summary['reduction_ratio']*100:.0f}%)")
print()

for c in summary['clusters']:
    print(f"  ┌─ Cluster [{c['cluster_id']}]")
    print(f"  │  Alert count:  {c['alert_count']}")
    print(f"  │  Services:     {c['services']}")
    print(f"  │  Time range:   {c['time_range'][0]} → {c['time_range'][1]}")
    print(f"  │  Max severity: {c['max_severity']}")
    print(f"  │  Alert IDs:    {c['alert_ids']}")
    print(f"  │  Fingerprints: {len(c['fingerprints'])} unique")
    for fp in c['fingerprints']:
        print(f"  │    • {fp}")
    print(f"  └{'─'*50}")
    print()

 CORRELATION RESULTS (gap_sec=120, max_hop=2)
  Input alerts:    20
  Output clusters: 3
  Reduction ratio: 0.85 (85%)

  ┌─ Cluster [c-000-000]
  │  Alert count:  18
  │  Services:     ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']
  │  Time range:   2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
  │  Max severity: crit
  │  Alert IDs:    ['a-0001', 'a-0002', 'a-0003', 'a-0004', 'a-0008', 'a-0011', 'a-0015', 'a-0018', 'a-0005', 'a-0006', 'a-0012', 'a-0017', 'a-0009', 'a-0007', 'a-0014', 'a-0020', 'a-0010', 'a-0019']
  │  Fingerprints: 15 unique
  │    • cart-svc|latency_p99_ms|warn
  │    • checkout-svc|downstream_payment_error_rate|crit
  │    • checkout-svc|latency_p99_ms|crit
  │    • checkout-svc|latency_p99_ms|warn
  │    • checkout-svc|request_drop_rate|crit
  │    • edge-lb|p99_latency_ms|crit
  │    • edge-lb|upstream_5xx_rate|crit
  │    • edge-lb|upstream_5xx_rate|warn
  │    • notification-svc|queue_depth|crit
  │    • notification-svc|queue_lag_ms|

## 8. Phân tích & Thảo luận

### Vì sao chỉ ra 1 cluster?

Có 2 lý do chính:

1. **Time-Window:** Tất cả 20 alert xảy ra trong khoảng 6.5 phút (09:42:01 → 09:48:30). Gap lớn nhất giữa 2 alert liên tiếp chỉ là 49 giây — rất nhỏ so với gap_sec=120. Do đó, tất cả rơi vào 1 session duy nhất.

2. **Topology (Union-Find transitive):** Với max_hop=2 và thuật toán Union-Find, các service bị gộp lại một cách bắc cầu (transitive). Ví dụ:
   - `edge-lb ↔ recommender-svc = 2 hops` (qua catalog-svc) → merge
   - `edge-lb ↔ checkout-svc = 1 hop` → merge
   - Kết quả: recommender-svc bị gộp vào cluster chính dù thực tế là lỗi KHÔNG liên quan (batch retrain).

### Đây chính là limitation lớn nhất của topology grouping!

In [8]:
# Phân tích alert a-0013 (recommender-svc) và a-0016 (search-svc)
print("=== Phân tích 2 Alert 'Noise' ===")
noise_alerts = [a for a in raw_alerts if a['id'] in ('a-0013', 'a-0016')]
for a in noise_alerts:
    print(f"\n  Alert {a['id']}:")
    print(f"    Service:  {a['service']}")
    print(f"    Metric:   {a['metric']}")
    print(f"    Severity: {a['severity']}")
    print(f"    Note:     {a['labels'].get('note', 'N/A')}")
    
    # Tính khoảng cách đến payment-svc (root cause giả định)
    try:
        dist = nx.shortest_path_length(undirected, a['service'], 'payment-svc')
        path = nx.shortest_path(undirected, a['service'], 'payment-svc')
        print(f"    Distance to payment-svc: {dist} hops")
        print(f"    Path: {' → '.join(path)}")
    except nx.NetworkXNoPath:
        print(f"    Distance to payment-svc: NO PATH")

print("\n" + "="*50)
print("KẾT LUẬN:")
print("  a-0013 (recommender-svc): KHÔNG liên quan đến payment cascade")
print("    → Note ghi rõ: 'unrelated — concurrent batch retrain'")
print("    → Nhưng correlator vẫn gộp vào cluster chính (FALSE POSITIVE)")
print()
print("  a-0016 (search-svc): KHÔNG liên quan đến payment cascade")
print("    → Note ghi rõ: 'noise — independent slow query'")
print("    → Nhưng correlator vẫn gộp vào cluster chính (FALSE POSITIVE)")

=== Phân tích 2 Alert 'Noise' ===

  Alert a-0013:
    Service:  recommender-svc
    Metric:   cpu_utilization
    Severity: warn
    Note:     unrelated — concurrent batch retrain
    Distance to payment-svc: 4 hops
    Path: recommender-svc → catalog-svc → edge-lb → checkout-svc → payment-svc

  Alert a-0016:
    Service:  search-svc
    Metric:   catalog_db_query_time_ms
    Severity: warn
    Note:     noise — independent slow query
    Distance to payment-svc: 3 hops
    Path: search-svc → edge-lb → checkout-svc → payment-svc

KẾT LUẬN:
  a-0013 (recommender-svc): KHÔNG liên quan đến payment cascade
    → Note ghi rõ: 'unrelated — concurrent batch retrain'
    → Nhưng correlator vẫn gộp vào cluster chính (FALSE POSITIVE)

  a-0016 (search-svc): KHÔNG liên quan đến payment cascade
    → Note ghi rõ: 'noise — independent slow query'
    → Nhưng correlator vẫn gộp vào cluster chính (FALSE POSITIVE)


## 9. Ghi kết quả ra file

In [9]:
# Ghi output ra results/cluster_summary.json
output_path = 'results/cluster_summary.json'
os.makedirs('results', exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"✅ Đã ghi kết quả vào: {output_path}")
print(f"   Input alerts:    {summary['input_alerts']}")
print(f"   Output clusters: {summary['output_clusters']}")
print(f"   Reduction ratio: {summary['reduction_ratio']}")
print()
print("=== File content (preview) ===")
print(json.dumps(summary, indent=2, ensure_ascii=False))

✅ Đã ghi kết quả vào: results/cluster_summary.json
   Input alerts:    20
   Output clusters: 3
   Reduction ratio: 0.85

=== File content (preview) ===
{
  "input_alerts": 20,
  "output_clusters": 3,
  "reduction_ratio": 0.85,
  "clusters": [
    {
      "cluster_id": "c-000-000",
      "alert_count": 18,
      "services": [
        "cart-svc",
        "checkout-svc",
        "edge-lb",
        "notification-svc",
        "payment-svc"
      ],
      "time_range": [
        "2026-06-12T09:42:01Z",
        "2026-06-12T09:48:30Z"
      ],
      "max_severity": "crit",
      "alert_ids": [
        "a-0001",
        "a-0002",
        "a-0003",
        "a-0004",
        "a-0008",
        "a-0011",
        "a-0015",
        "a-0018",
        "a-0005",
        "a-0006",
        "a-0012",
        "a-0017",
        "a-0009",
        "a-0007",
        "a-0014",
        "a-0020",
        "a-0010",
        "a-0019"
      ],
      "fingerprints": [
        "cart-svc|latency_p99_ms|warn",
        "